# 16. 🚀 Продвинутое: теория обучения и обобщение

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/16_learning_theory.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m16). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**16.3.** Проверьте численно, что GD из нуля на недоопределённой регрессии приходит к решению минимальной нормы.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(size=(20, 100)); y = rng.normal(size=20)       # d = 100 > N = 20
w = np.zeros(100)
for _ in range(20_000):
    w -= 0.005 * X.T @ (X @ w - y)
w_min = np.linalg.pinv(X) @ y
print(np.abs(X @ w - y).max(), np.linalg.norm(w - w_min))   # интерполирует и совпадает с pinv

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def double_descent():
    rng = np.random.default_rng(1)
    n = 40
    f = lambda x: np.sin(2 * np.pi * x)
    xtr = rng.uniform(-1, 1, n); ytr = f(xtr) + 0.2 * rng.normal(size=n)
    xte = rng.uniform(-1, 1, 2000); yte = f(xte) + 0.2 * rng.normal(size=2000)
    widths = np.unique(np.r_[np.arange(4, 60, 4), np.geomspace(60, 3000, 14).astype(int)])
    te_err = []
    for p in widths:
        errs = []
        for seed in range(60):
            r = np.random.default_rng(seed)
            a = r.normal(size=p); b = r.uniform(-1, 1, p) * np.abs(a)
            phi = lambda x: np.maximum(0, np.outer(x, a) + b)        # случайные ReLU-признаки
            w = np.linalg.pinv(phi(xtr)) @ ytr                       # решение минимальной нормы
            errs.append(np.mean((phi(xte) @ w - yte) ** 2))
        te_err.append(np.median(errs))
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(widths, te_err, "o-", color=C[1], ms=4, lw=1.8)
    ax.axvline(n, color="k", ls="--", lw=1)
    ax.text(n * 1.08, max(te_err) * 0.8, "порог интерполяции:\nпараметров ≈ N = 40", fontsize=9)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("число случайных ReLU-признаков (лог. шкала)"); ax.set_ylabel("test MSE (медиана по 60 запускам)")
    ax.set_title("Double descent: пик у порога интерполяции, дальше ошибка снова падает")
    save(fig, "double_descent")

double_descent()